# Notebook 3 — Exploratory Data Analysis (EDA)
### CDC Foodborne Disease Outbreak Surveillance System (1998–2015)

**Research questions:** How do recorded outbreak frequency, illness burden and *reported* hospitalization/fatality proportions vary by year, pathogen, recorded food category and setting?

**Dataset:** 18,839 retained source records × 12 columns · 1998–2015 · original 12-column deduplication policy.
**Input:** `data/cleaned_data_grouped.csv` (NB2 output).
**Outcome policy:** Hospitalizations/Fatalities not reported in the source are preserved as `NaN`, never assumed zero. Each observed-outcome proportion uses the illness denominator of records with that particular outcome observed. It is **not** an exposure-adjusted risk estimate.

**Study limitations:** These are surveillance records, not independently verified unique epidemiological events; the first-listed location is an analytical proxy, and grouped food/pathogen labels simplify heterogeneous data. Descriptive associations cannot establish causes, clinical prognosis or regulatory efficacy.

**Analysis structure:** Univariate frequencies → setting comparisons → associations and pathogen burden → food × pathogen aggregation.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# ── Global style ──────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})
PALETTE_MAIN  = "rocket_r"
PALETTE_COOL  = "mako_r"
PALETTE_DIVG  = "RdYlGn_r"
COLOR_SINGLE  = "#2E6FA3"
COLOR_DANGER  = "#C0392B"

In [ ]:
df = pd.read_csv("data/cleaned_data_grouped.csv")
assert df.shape == (18839, 12), f"Unexpected grouped dataset: {df.shape}"
assert df[["Hospitalizations", "Fatalities"]].isna().sum().to_dict() == {
    "Hospitalizations": 3560, "Fatalities": 3538
}, "Outcome missingness was not preserved"
print("Unreported outcome values remain NaN; use matched observed denominators.")

# Month as ordered categorical (used throughout)
MONTH_ORDER = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
df["Month"] = pd.Categorical(df["Month"], categories=MONTH_ORDER, ordered=True)

print(f"Dataset: {df.shape[0]:,} records × {df.shape[1]} columns")
print(f"Years: {df['Year'].min()} – {df['Year'].max()}")
print(f"States: {df['State'].nunique()}  |  "
      f"Food categories: {df['Food_grouped'].nunique()}  |  "
      f"Species categories: {df['Species_grouped'].nunique()}")
print(f"Primary location types: {df['primary_location'].nunique()} "
      f"(normalised in 02_entity_normalisation.ipynb)")
df.head(3)

## 2. Helper functions

Centralised reusable plotting utilities — keeps all analysis cells concise.

In [ ]:
def plot_bar(data, x, y, title, xlabel="", ylabel="",
             palette=PALETTE_MAIN, rotate=0, figsize=(11, 5), top_n=None):
    """Ranked horizontal or vertical bar chart with value annotations."""
    if top_n:
        data = data.nlargest(top_n, y)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(data=data, x=x, y=y, palette=palette, ax=ax, order=data[x])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if rotate:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=rotate, ha="right")
    # value labels
    for p in ax.patches:
        val = p.get_height()
        if val > 0:
            ax.annotate(f"{val:,.0f}", (p.get_x() + p.get_width()/2, val),
                        ha="center", va="bottom", fontsize=8.5, color="#444")
    plt.tight_layout()
    plt.show()

def plot_hbar(data, y_col, x_col, title, palette=COLOR_SINGLE, figsize=(10, 6)):
    """Horizontal bar chart – ideal for long category labels."""
    fig, ax = plt.subplots(figsize=figsize)
    colors = [COLOR_DANGER if i == 0 else palette for i in range(len(data))]
    ax.barh(data[y_col], data[x_col], color=colors)
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel("")
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    plt.tight_layout()
    plt.show()

def exclude_unknown(data, col):
    """Drop 'Unknown' rows for cleaner ranked visualisations."""
    return data[data[col].str.lower() != "unknown"].copy()

TOP_N = 10  # default top-N for all ranked charts

def observed_outcome_summary(frame, group_col, outcome):
    """Outcome-specific reported proportion and observation coverage per group.

    Outcome numerator and illness denominator use the SAME observed rows.
    Reporting coverage is shown separately; missingness may be selective.
    """
    if outcome not in ("Hospitalizations", "Fatalities"):
        raise ValueError("Invalid outcome")
    base = frame.groupby(group_col, observed=True, dropna=False).agg(
        outbreaks=("Illnesses", "size"),
        total_ill=("Illnesses", "sum"),
        avg_size=("Illnesses", "mean"),
    )
    observed = frame[frame[outcome].notna() & frame["Illnesses"].gt(0)]
    obs = observed.groupby(group_col, observed=True, dropna=False).agg(
        recorded=(outcome, "sum"),
        observed_ill=("Illnesses", "sum"),
        observed_outbreaks=("Illnesses", "size"),
    )
    summary = base.join(obs)
    summary["observed_outbreaks"] = summary["observed_outbreaks"].fillna(0).astype(int)
    summary["reporting_coverage"] = summary["observed_outbreaks"] / summary["outbreaks"]
    summary["reported_proportion"] = summary["recorded"] / summary["observed_ill"].replace(0, np.nan)
    return summary


def pathogen_severity_table(frame):
    """Separate observed hospitalization/fatality denominators for each pathogen."""
    hosp = observed_outcome_summary(frame, "Species_grouped", "Hospitalizations")
    fatal = observed_outcome_summary(frame, "Species_grouped", "Fatalities")
    result = hosp.rename(columns={
        "recorded":"Hospitalizations", "observed_ill":"Hosp_ill_observed",
        "observed_outbreaks":"Hosp_outbreaks_observed",
        "reported_proportion":"Hosp_rate", "reporting_coverage":"Hosp_coverage"
    })
    result = result.join(fatal[["recorded","observed_ill","observed_outbreaks",
                               "reported_proportion","reporting_coverage"]].rename(columns={
        "recorded":"Fatalities", "observed_ill":"Fatal_ill_observed",
        "observed_outbreaks":"Fatal_outbreaks_observed",
        "reported_proportion":"Fatal_rate", "reporting_coverage":"Fatal_coverage"
    }))
    result["Illnesses"] = result["total_ill"]
    return result.reset_index()


## 3. Univariate analysis

Examining each variable independently to establish baseline distributions before
looking at relationships.

### 3.1 Temporal patterns — outbreaks by year and month

In [ ]:
# ── Year ──────────────────────────────────────────────────────
df_year = df["Year"].value_counts().reset_index()
df_year.columns = ["Year", "Outbreaks"]
df_year = df_year.sort_values("Year")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar
sns.barplot(data=df_year, x="Year", y="Outbreaks",
            palette="rocket_r", ax=axes[0])
axes[0].set_title("Outbreaks per year")
axes[0].set_xlabel("Year")
axes[0].tick_params(axis="x", rotation=45)

# Trend line
axes[1].plot(df_year["Year"], df_year["Outbreaks"],
             marker="o", color=COLOR_SINGLE, linewidth=2, markersize=5)
axes[1].axhline(df_year["Outbreaks"].mean(), color="grey",
                linestyle="--", linewidth=1, label="Mean")
axes[1].set_title("Outbreak trend (1998–2015)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Outbreaks")
axes[1].legend()

plt.suptitle("Annual outbreak frequency", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

peak_year = df_year.loc[df_year["Outbreaks"].idxmax(), "Year"]
low_year  = df_year.loc[df_year["Outbreaks"].idxmin(), "Year"]
print(f"Peak year: {peak_year} ({df_year['Outbreaks'].max()} outbreaks)")
print(f"Lowest year: {low_year} ({df_year['Outbreaks'].min()} outbreaks)")
print(f"Overall trend: {df_year['Outbreaks'].iloc[0]} (1998) → {df_year['Outbreaks'].iloc[-1]} (2015)"
      f"  ({(df_year['Outbreaks'].iloc[-1]/df_year['Outbreaks'].iloc[0]-1)*100:.0f}% change)")

**Observation:** Recorded annual outbreak counts vary over time. An endpoint comparison alone does not establish a monotonic decline or its causes. Changes in exposure, reporting, surveillance capacity and regulation cannot be separated from these aggregate records.

In [ ]:
# ── Month ─────────────────────────────────────────────────────
df_month = df["Month"].value_counts().reset_index()
df_month.columns = ["Month", "Outbreaks"]
df_month = df_month.sort_values("Month")

fig, ax = plt.subplots(figsize=(11, 5))
bars = sns.barplot(data=df_month, x="Month", y="Outbreaks",
                   palette="mako_r", ax=ax, order=MONTH_ORDER)
ax.set_title("Seasonal distribution of outbreaks (1998–2015)")
ax.set_xlabel("Month")
ax.set_ylabel("Total outbreaks")
ax.tick_params(axis="x", rotation=40)

# Highlight summer and winter peaks
peak_months = ["May", "June", "December"]
for patch, label in zip(ax.patches, MONTH_ORDER):
    if label in peak_months:
        patch.set_edgecolor(COLOR_DANGER)
        patch.set_linewidth(2.5)
plt.tight_layout()
plt.show()

**Observation:** Monthly reported outbreak counts show seasonality. Biological growth, consumer practices, reporting patterns and seasonal exposures are *possible* explanations, not mechanisms identified by this dataset. Lower reported counts cannot be translated into lower population exposure risk.

### 3.2 Top food vehicles and pathogens by outbreak frequency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Food
food_counts = (exclude_unknown(df, "Food_grouped")["Food_grouped"]
               .value_counts().head(TOP_N).reset_index())
food_counts.columns = ["Food", "Outbreaks"]
sns.barplot(data=food_counts, x="Outbreaks", y="Food",
            palette=PALETTE_MAIN, ax=axes[0])
axes[0].set_title(f"Top {TOP_N} food vehicles")
axes[0].set_xlabel("Number of outbreaks")
axes[0].set_ylabel("")

# Pathogen
path_counts = (exclude_unknown(df, "Species_grouped")["Species_grouped"]
               .value_counts().head(TOP_N).reset_index())
path_counts.columns = ["Pathogen", "Outbreaks"]
sns.barplot(data=path_counts, x="Outbreaks", y="Pathogen",
            palette=PALETTE_COOL, ax=axes[1])
axes[1].set_title(f"Top {TOP_N} pathogens by outbreak count")
axes[1].set_xlabel("Number of outbreaks")
axes[1].set_ylabel("")

plt.suptitle("What causes most outbreaks? (frequency, not severity)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Top 3 food vehicles:", food_counts["Food"].tolist()[:3])
print("Top 3 pathogens:    ", path_counts["Pathogen"].tolist()[:3])

**Interpretation:** The food and pathogen ranking concerns reported outbreaks and analytical grouping labels. Mixed-ingredient dishes can have uncertain primary food attribution. Higher recorded frequency is not a direct estimate of contamination probability, food-specific risk or biological transmission mechanism.

### 3.3 Geographic and setting distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── State ─────────────────────────────────────────────────────
state_counts = df["State"].value_counts().head(15).reset_index()
state_counts.columns = ["State", "Outbreaks"]
sns.barplot(data=state_counts, x="State", y="Outbreaks",
            palette=PALETTE_MAIN, ax=axes[0])
axes[0].set_title("Top 15 states by outbreak count")
axes[0].tick_params(axis="x", rotation=60)
for p in axes[0].patches:
    v = p.get_height()
    if v > 50:
        axes[0].annotate(f"{v:,.0f}", (p.get_x()+p.get_width()/2, v),
                         ha="center", va="bottom", fontsize=8, color="#444")

# ── Primary location type ─────────────────────────────────────
loc_counts = (exclude_unknown(df, "primary_location")["primary_location"]
              .value_counts().head(12).reset_index())
loc_counts.columns = ["Location", "Outbreaks"]
sns.barplot(data=loc_counts, x="Outbreaks", y="Location",
            palette=PALETTE_COOL, ax=axes[1])
axes[1].set_title("Top 12 outbreak settings (primary venue)")
axes[1].set_xlabel("Number of outbreaks")
axes[1].set_ylabel("")

plt.suptitle("Where do outbreaks occur? — frequency view",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

top3_states = state_counts["State"].tolist()[:3]
print(f"Top 3 states: {top3_states}")
print(f"Restaurant share of all outbreaks: "
      f"{(df['primary_location']=='Restaurant').sum()/len(df)*100:.1f}%")

**Observation:** The state and first-listed-setting charts rank numbers of *recorded* outbreaks, without denominators for population, meals or exposure. Florida's ranking cannot be causally attributed to its climate, tourism, seafood consumption or regulatory conditions using these data. First-listed settings are analytical proxies rather than verified exposure sites.

### 3.4 Setting severity profile — where frequency meets clinical risk

In [ ]:
# ── Setting profile with outcome-specific observed denominators ─────
eligible = df[df["primary_location"].ne("Unknown")].copy()
h = observed_outcome_summary(eligible, "primary_location", "Hospitalizations")
f = observed_outcome_summary(eligible, "primary_location", "Fatalities")
loc_profile = h.rename(columns={
    "outbreaks":"outbreaks", "total_ill":"total_ill", "avg_size":"avg_size",
    "recorded":"total_hosp", "observed_ill":"hosp_observed_ill",
    "observed_outbreaks":"hosp_observed_outbreaks",
    "reporting_coverage":"hosp_coverage", "reported_proportion":"hosp_rate",
}).join(f[["recorded","observed_ill","observed_outbreaks",
          "reporting_coverage","reported_proportion"]].rename(columns={
    "recorded":"total_fatal", "observed_ill":"fatal_observed_ill",
    "observed_outbreaks":"fatal_observed_outbreaks",
    "reporting_coverage":"fatal_coverage", "reported_proportion":"fatal_rate",
})).reset_index()

# Descriptive threshold for legibility, not statistical validation.
loc_stable = loc_profile[(loc_profile["outbreaks"] >= 30) &
                         (loc_profile["hosp_observed_outbreaks"] >= 30) &
                         loc_profile["hosp_observed_ill"].gt(0)].copy()

fig, ax = plt.subplots(figsize=(13, 7))
bubble_size = loc_stable["outbreaks"] / loc_stable["outbreaks"].max() * 1800 + 80
ax.scatter(loc_stable["avg_size"], loc_stable["hosp_rate"],
           s=bubble_size, color=COLOR_SINGLE, alpha=0.65, edgecolors="white")
for _, row in loc_stable.iterrows():
    ax.annotate(str(row["primary_location"]).replace("/", "/\\n"),
                (row["avg_size"], row["hosp_rate"]),
                xytext=(5, 3), textcoords="offset points", fontsize=8)
ax.set(xlabel="Mean illnesses per recorded outbreak",
       ylabel="Recorded hospitalizations / illnesses in hospitalization-reported outbreaks",
       title="Setting comparison: reported hospitalization proportions (observed outcomes)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.show()

focus = ["Restaurant", "Private Home/Residence", "Prison/Jail",
         "Nursing Home/Assisted Living Facility"]
display(loc_profile[loc_profile["primary_location"].isin(focus)][[
    "primary_location", "outbreaks", "total_ill", "avg_size",
    "total_hosp", "hosp_observed_ill", "hosp_observed_outbreaks",
    "hosp_coverage", "hosp_rate", "total_fatal", "fatal_observed_ill",
    "fatal_observed_outbreaks", "fatal_coverage", "fatal_rate"
]].round(4))

**Setting comparison — observed data only.** The following outcomes are reported proportions in the source records, **not** exposure-adjusted clinical risk estimates.

| First-listed setting | Records | Mean illnesses/record | Hospitalization proportion (observed outcome) | Recorded deaths |
| --- | ---: | ---: | ---: | ---: |
| Restaurant | 10,843 | 14.0 | 4.43% | 74 |
| Prison/Jail | 193 | 106.8 | 0.97% | 4 |
| Private Home/Residence | 1,982 | 15.6 | 10.49% | 85 |
| Nursing Home/Assisted Living Facility | 186 | 30.5 | 6.61% | 44 |

Interpretation is limited by selective outcome missingness, first-listed venue proxies, category definitions and lack of exposure denominators. These descriptive contrasts do not determine setting-specific regulatory priority, vulnerability or causality. The preceding code displays outcome-specific observation counts and denominators.

In [ ]:
# ── Setting severity comparison — ranked by metric ────────────
# Descriptive setting comparison; outcome-specific observed denominators
SHOW_LOCS = [
    "Restaurant", "Fast Food Restaurant", "Catering Service",
    "Banquet Facility", "Private Home/Residence",
    "School/College/University", "Prison/Jail",
    "Nursing Home/Assisted Living Facility", "Hospital",
]

subset = loc_profile[(loc_profile["primary_location"].isin(SHOW_LOCS)) & (loc_profile["outbreaks"] >= 30) & (loc_profile["hosp_observed_outbreaks"] >= 30) & (loc_profile["fatal_observed_outbreaks"] >= 30)].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("avg_size",   "Average outbreak size (cases)", False),
    ("hosp_rate",  "Hospitalisation rate",           True),
    ("fatal_rate", "Fatality rate",                  True),
]

for ax, (col, label, pct) in zip(axes, metrics):
    sub = subset.sort_values(col, ascending=True)
    # Colour by observed value — top 3 highlighted, no pre-assigned categories
    max_val = sub[col].max()
    colors = [COLOR_DANGER if v == max_val else COLOR_SINGLE for v in sub[col]]
    bars = ax.barh(sub["primary_location"], sub[col], color=colors)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.invert_yaxis()
    if pct:
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

plt.suptitle("Setting outcomes — observed data and descriptive eligibility filters",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3.5 Setting × pathogen: recorded composition, not exposure causation

In [ ]:
# ── Location × Pathogen heatmap ──────────────────────────────
ANALYSIS_LOCS = [
    "Restaurant",
    "Fast Food Restaurant",
    "Catering Service",
    "Private Home/Residence",
    "School/College/University",
    "Prison/Jail",
    "Nursing Home/Assisted Living Facility",
    "Banquet Facility",
]
top8_path = (exclude_unknown(df, "Species_grouped")["Species_grouped"]
             .value_counts().head(8).index.tolist())

df_lp = df[
    df["primary_location"].isin(ANALYSIS_LOCS) &
    df["Species_grouped"].isin(top8_path)
].copy()

# Two views: outbreak count and illness burden
hm_count = pd.crosstab(df_lp["primary_location"], df_lp["Species_grouped"])
hm_ill   = df_lp.pivot_table(
    values="Illnesses", index="primary_location",
    columns="Species_grouped", aggfunc="sum", fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(hm_count, cmap="YlOrRd", annot=True, fmt="d",
            linewidths=0.4, linecolor="#ddd", ax=axes[0])
axes[0].set_title("Outbreak count (setting × pathogen)")
axes[0].set_xlabel("Pathogen")
axes[0].set_ylabel("Setting")
axes[0].tick_params(axis="x", rotation=40)

sns.heatmap(hm_ill, cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.4, linecolor="#ddd", ax=axes[1])
axes[1].set_title("Total illnesses (setting × pathogen)")
axes[1].set_xlabel("Pathogen")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=40)

plt.suptitle("Setting × pathogen risk matrix\n"
             "Pathogen composition varies by setting — Norovirus dominant except in Prison and Private Home",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Observation:** The pathogen composition varies among first-listed setting labels in these surveillance records. The tabulations are conditioned on selected known pathogen categories and do not establish where infection originated, what caused transmission or whether a setting is inherently more dangerous. The counts and illness totals should be read directly from the charts rather than an older fixed summary table.

In [ ]:
# ── Pathogen proportion per setting (normalised) ─────────────
hm_prop = hm_count.div(hm_count.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(14, 6))
hm_prop.plot(kind="bar", stacked=True, colormap="tab10",
             ax=ax, width=0.75, edgecolor="white")
ax.set_title("Pathogen composition by setting (proportion)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Setting")
ax.set_ylabel("Proportion of outbreaks")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="Pathogen", bbox_to_anchor=(1.01, 1),
          loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**Observation:** The stacked chart shows the proportion of the selected known pathogens within each first-listed setting category. Other pathogens and unknown species are omitted. This is not the pathogen distribution among all possible cases or a causal explanation of location differences.

## 4. Bivariate analysis

### 4.1 Numerical × Numerical — illness burden and severity over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
,
,# Total illnesses per year
,df_ill = df.groupby("Year")["Illnesses"].sum().reset_index()
,axes[0].bar(df_ill["Year"], df_ill["Illnesses"], color=COLOR_SINGLE, alpha=0.85)
,axes[0].set_title("Total illnesses per year")
,axes[0].set_xlabel("Year")
,axes[0].set_ylabel("Total illnesses")
,axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
,axes[0].tick_params(axis="x", rotation=45)
,
,# Scatter: outbreak size vs hospitalisation
,observed_hosp = df[df["Hospitalizations"].notna() & df["Illnesses"].gt(0)].copy()
r = observed_hosp["Illnesses"].corr(observed_hosp["Hospitalizations"])
,axes[1].scatter(observed_hosp["Illnesses"], observed_hosp["Hospitalizations"],
,                alpha=0.25, s=15, color=COLOR_SINGLE)
,m, b = np.polyfit(observed_hosp["Illnesses"], observed_hosp["Hospitalizations"], 1)
,x_line = np.linspace(0, observed_hosp["Illnesses"].max(), 100)
,axes[1].plot(x_line, m * x_line + b, color=COLOR_DANGER, linewidth=2,
,             label=f"Regression  (r = {r:.2f})")
,axes[1].set_title("Outbreak size vs hospitalisations")
,axes[1].set_xlabel("Illnesses per outbreak")
,axes[1].set_ylabel("Hospitalisations per outbreak")
,axes[1].legend()
,
,plt.tight_layout()
,plt.show()
,
,print(f"Pearson r (Illnesses vs Hospitalisations): {r:.3f}")
,print(f"Observed hospitalization records: {len(observed_hosp):,} / {len(df):,}")
,print("Association only; cannot rank causal/predictive importance without a model and missingness analysis.")

**Observation:** The plot and Pearson correlation are computed **only for outbreaks with recorded hospitalization outcomes**, not after imputing unreported values to zero. Correlation does not establish a causal effect or compare the predictive importance of outbreak size versus pathogen identity. Missing outcome reporting may introduce selection bias.

### 4.2 Categorical × Numerical — pathogen burden and severity

In [ ]:
# ── Illnesses vs Hospitalisations: two different signals ─────
species_ill  = (df.groupby("Species_grouped")["Illnesses"].sum()
                .sort_values(ascending=False))
species_hosp = (df.groupby("Species_grouped")["Hospitalizations"].sum()
                .sort_values(ascending=False))

top_ill  = exclude_unknown(species_ill.reset_index().head(TOP_N+1),
                           "Species_grouped").head(TOP_N)
top_hosp = exclude_unknown(species_hosp.reset_index().head(TOP_N+1),
                           "Species_grouped").head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=top_ill, x="Illnesses", y="Species_grouped",
            palette=PALETTE_MAIN, ax=axes[0])
axes[0].set_title("Total illnesses (1998–2015)")
axes[0].set_xlabel("Total illnesses")
axes[0].set_ylabel("")

sns.barplot(data=top_hosp, x="Hospitalizations", y="Species_grouped",
            palette=PALETTE_COOL, ax=axes[1])
axes[1].set_title("Total hospitalisations (1998–2015)")
axes[1].set_xlabel("Total hospitalisations")
axes[1].set_ylabel("")

plt.suptitle("Illness burden vs hospitalisation burden — different rankings",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

ill_rank  = top_ill["Species_grouped"].tolist()[:3]
hosp_rank = top_hosp["Species_grouped"].tolist()[:3]
print(f"Top 3 by illness:          {ill_rank}")
print(f"Top 3 by hospitalisation:  {hosp_rank}")
print()
norovirus_ill  = int(species_ill.get("Norovirus", 0))
salmon_hosp    = int(species_hosp.iloc[0])
print(f"Norovirus total illnesses:  {norovirus_ill:,} ({norovirus_ill/species_ill.sum()*100:.1f}% of total)")
print(f"Salmonella total hosp.:     {salmon_hosp:,} ({salmon_hosp/species_hosp.sum()*100:.1f}% of total)")

**Recorded burden versus reported proportion:** Norovirus has the largest recorded illness total among known pathogen groups; *Salmonella enterica* has the largest recorded hospitalization sum among known pathogens. These are descriptive burden rankings, not population-relative clinical risk or causal evidence. Hospitalization counts with unknown values excluded should be labeled **recorded hospitalizations**, and the corresponding observation coverage should be reported when proportions are used.

In [ ]:
# Hospitalization and fatality proportions have separate observed denominators.
severity = pathogen_severity_table(df[df["Illnesses"] > 0])
top_hosp_rate = (exclude_unknown(severity, "Species_grouped")
                 .dropna(subset=["Hosp_rate"]).nlargest(TOP_N, "Hosp_rate"))
top_fatal_rate = (exclude_unknown(severity, "Species_grouped")
                  .dropna(subset=["Fatal_rate"]).nlargest(TOP_N, "Fatal_rate"))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(top_hosp_rate["Species_grouped"], top_hosp_rate["Hosp_rate"], color=COLOR_SINGLE)
axes[0].invert_yaxis()
axes[0].set_title("Reported hospitalization proportion by pathogen (unfiltered)")
axes[0].set_xlabel("Recorded hospitalizations / illnesses in reporting outbreaks")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].barh(top_fatal_rate["Species_grouped"], top_fatal_rate["Fatal_rate"], color=COLOR_SINGLE)
axes[1].invert_yaxis()
axes[1].set_title("Reported fatality proportion by pathogen (unfiltered)")
axes[1].set_xlabel("Recorded deaths / illnesses in reporting outbreaks")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()
display(severity[severity["Species_grouped"].isin(
    ["Norovirus", "Salmonella enterica", "Listeria monocytogenes"]
)][["Species_grouped", "Illnesses", "Hospitalizations", "Hosp_ill_observed",
    "Hosp_outbreaks_observed", "Hosp_coverage", "Hosp_rate", "Fatalities",
    "Fatal_ill_observed", "Fatal_outbreaks_observed", "Fatal_coverage", "Fatal_rate"]].round(4))

**Interpretation:** Unfiltered observed-outcome proportions for rare pathogens may be extremely unstable because few outbreaks report outcomes. Even a large observed proportion is not an estimate of disease-level prognosis or population risk. Listeria's observed-outcome fatality proportion must be reported together with its observed denominator and reporting coverage.

In [ ]:
# Descriptive eligibility threshold, not proof of statistical stability.
# Apply each outcome-specific minimum to its OWN observed-illness denominator.
MIN_OBS_ILL = 30
known_severity = exclude_unknown(severity, "Species_grouped")
top_hosp_stable = known_severity[
    known_severity["Hosp_ill_observed"].ge(MIN_OBS_ILL) &
    known_severity["Hosp_outbreaks_observed"].gt(0)
].dropna(subset=["Hosp_rate"]).nlargest(TOP_N, "Hosp_rate")
top_fatal_stable = known_severity[
    known_severity["Fatal_ill_observed"].ge(MIN_OBS_ILL) &
    known_severity["Fatal_outbreaks_observed"].gt(0)
].dropna(subset=["Fatal_rate"]).nlargest(TOP_N, "Fatal_rate")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(top_hosp_stable["Species_grouped"],
             top_hosp_stable["Hosp_rate"], color=COLOR_SINGLE)
axes[0].invert_yaxis()
axes[0].set_title("Reported hospitalization proportion (≥30 observed illnesses)")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].barh(top_fatal_stable["Species_grouped"],
             top_fatal_stable["Fatal_rate"], color=COLOR_SINGLE)
axes[1].invert_yaxis()
axes[1].set_title("Reported fatality proportion (≥30 observed illnesses)")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()
print("Hospitalization ranking:");display(top_hosp_stable[[
    "Species_grouped", "Hosp_rate", "Hosp_ill_observed", "Hosp_outbreaks_observed", "Hosp_coverage"
]])
print("Fatality ranking:");display(top_fatal_stable[[
    "Species_grouped", "Fatal_rate", "Fatal_ill_observed", "Fatal_outbreaks_observed", "Fatal_coverage"
]])

**Descriptive filter caution:** Requiring at least 30 illnesses from outbreaks where the relevant outcome is reported reduces extreme small-denominator displays, but does not itself validate statistical stability, eliminate missing-not-at-random bias or enable comparison with population clinical risk. Hospitalization and fatality filters apply to distinct observed subsets.

### 4.3 Temporal × pathogen — are seasonal and annual compositions changing?

In [ ]:
top10_path = (exclude_unknown(df, "Species_grouped")["Species_grouped"]
              .value_counts().head(TOP_N).index.tolist())
df_top10 = df[df["Species_grouped"].isin(top10_path)].copy()
df_top10["Month"] = pd.Categorical(df_top10["Month"],
                                    categories=MONTH_ORDER, ordered=True)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Annual proportion
ct_year = pd.crosstab(df_top10["Year"], df_top10["Species_grouped"])
ct_year_prop = ct_year.div(ct_year.sum(axis=1), axis=0)
ct_year_prop.plot(kind="bar", stacked=True, colormap="tab10",
                  ax=axes[0], width=0.85)
axes[0].set_title("Annual pathogen composition (proportion)")
axes[0].set_ylabel("Proportion of outbreaks")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Pathogen", bbox_to_anchor=(1.01, 1), loc="upper left",
               fontsize=8)

# Seasonal proportion
ct_month = pd.crosstab(df_top10["Month"], df_top10["Species_grouped"])
ct_month_prop = ct_month.div(ct_month.sum(axis=1), axis=0)
ct_month_prop.plot(kind="bar", stacked=True, colormap="tab10",
                   ax=axes[1], width=0.85)
axes[1].set_title("Seasonal pathogen composition (proportion)")
axes[1].set_ylabel("Proportion of outbreaks")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="Pathogen", bbox_to_anchor=(1.01, 1), loc="upper left",
               fontsize=8)

plt.suptitle("Pathogen composition: stable over time, variable by season",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**Observation:** The annual and monthly stacked bars describe the composition of selected reported pathogen groups; they do not test whether changing refrigeration standards, environmental persistence or public-health actions caused the patterns. Unknown pathogens and other unselected groups are excluded from these proportions.

### 4.4 Categorical × Categorical — recorded food–pathogen counts and illness burden

In [ ]:
top10_food = (exclude_unknown(df, "Food_grouped")["Food_grouped"]
              .value_counts().head(TOP_N).index.tolist())

df_matrix = df[
    df["Species_grouped"].isin(top10_path) &
    df["Food_grouped"].isin(top10_food)
].copy()

# Pivot: illnesses among selected known groups (not a severity-adjusted or risk measure)
heatmap_ill = df_matrix.pivot_table(
    values="Illnesses",
    index="Species_grouped",
    columns="Food_grouped",
    aggfunc="sum",
    fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Outbreak count heatmap
hm_count = pd.crosstab(df_matrix["Species_grouped"], df_matrix["Food_grouped"])
sns.heatmap(hm_count, cmap="YlOrRd", annot=True, fmt="d",
            linewidths=0.4, linecolor="#ddd", ax=axes[0])
axes[0].set_title("Outbreak count (food × pathogen)")
axes[0].set_xlabel("Food vehicle")
axes[0].set_ylabel("Pathogen")
axes[0].tick_params(axis="x", rotation=40)

# Illness burden heatmap
sns.heatmap(heatmap_ill, cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.4, linecolor="#ddd", ax=axes[1])
axes[1].set_title("Total illnesses (food × pathogen)")
axes[1].set_xlabel("Food vehicle")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=40)

plt.suptitle("Food–pathogen recorded count vs illness burden",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Top combinations
top_combos = (df_matrix.groupby(["Species_grouped","Food_grouped"])["Illnesses"]
              .sum().sort_values(ascending=False).head(5))
print("Top 5 among preselected known food/pathogen groups:")
print(top_combos.to_string())

**Key descriptive finding:** Under the project's existing Food/Species grouping, Norovirus × Salad has 10,835 recorded illnesses and is the highest illness-burden pair **when both food and pathogen are known**. Unknown-category pairs are larger in the unrestricted ranking. This does not establish that salad caused the infections or that this pair has the highest exposure-adjusted risk. Proposed transmission mechanisms, process-control failures and interventions require independent evidence.

## 5. Summary of verified descriptive findings (revised 18,839-record cohort)

| Dimension | Verified result / interpretation |
| --- | --- |
| Analysis cohort | 18,839 retained source records after original 12-column exact-row deduplication |
| Total illnesses | 372,135 recorded |
| Recorded hospitalizations/deaths | 14,669 / 336; unreported severity outcomes remain missing |
| Overall observed hospitalization proportion | 14,669 / 303,446 = **4.834%**, using hospitalization-reported outbreak illnesses |
| Overall observed fatality proportion | 336 / 306,342 = **0.110%**, using fatality-reported outbreak illnesses |
| Norovirus illness burden | 137,651 / 372,135 = **36.99%**, based on existing grouped labels |
| Salmonella enterica hospitalization burden | 7,417 / 14,669 = **50.56%** of recorded hospitalizations |
| Listeria monocytogenes hospitalization proportion | 576 / 657 = **87.67%**, among hospitalization-reported outbreaks |
| Listeria monocytogenes fatality proportion | 121 / 711 = **17.02%**, among fatality-reported outbreaks |
| Private Home/Residence hospitalization proportion | 2,889 / 27,547 = **10.49%**, outcome-reported subset |
| Restaurant hospitalization proportion | 5,534 / 124,819 = **4.43%**, outcome-reported subset |
| Nursing Home/Assisted Living Facility | 44 recorded fatalities; 44 / 4,547 = **0.97%** observed fatality proportion |
| Food × pathogen | Norovirus × Salad = **10,835 recorded illnesses**, highest among known/known pairs under existing grouping |

**Interpretive boundaries:** One source record does not necessarily represent a unique epidemiological event. Source-reported outcomes may be selectively missing. Outcome-specific observed proportions are not population risk or clinical prognosis. Food/venue labels are analytical proxies, not proof of contamination or causation. Apparent seasonality and state differences cannot establish regulatory effects.

**Next step:** Align NB4 SQL and the public Streamlit dashboard with this changed cohort and outcome-specific denominators; do not display old hard-coded severity results as current.